In [1]:
import pandas as pd
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from scipy.stats import mode
import time
import os
from joblib import dump, load
import os
from scipy.stats import mode
from tqdm import tqdm
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
    

SEED = 42
NUM_FEATURES = 13

import pandas as pd
df = pd.read_csv('train_test_network.csv', encoding='utf-8')

In [2]:
categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"\nColumn: {col}")
    print(dict(zip(le.classes_, le.transform(le.classes_))))

feature_names = df.columns

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Original class distribution: {np.bincount(y_train)}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


pca = PCA(n_components=NUM_FEATURES, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test_scaled)

print("Features Selected: ")
for i, component in enumerate(pca.components_):
    # Get the index of the feature with the largest absolute value in this component
    important_idx = np.argmax(np.abs(component))
    print(f"Principal Component {i+1}: most important feature is '{feature_names[important_idx]}'")


X_train = X_train_pca
X_test = X_test_pca

print(X_train.shape)
print(X_test.shape)



Column: src_ip
{'13.55.50.68': 0, '172.17.0.5': 1, '172.17.0.7': 2, '172.17.0.8': 3, '192.168.1.1': 4, '192.168.1.103': 5, '192.168.1.133': 6, '192.168.1.152': 7, '192.168.1.169': 8, '192.168.1.17': 9, '192.168.1.180': 10, '192.168.1.184': 11, '192.168.1.186': 12, '192.168.1.190': 13, '192.168.1.191': 14, '192.168.1.192': 15, '192.168.1.193': 16, '192.168.1.194': 17, '192.168.1.195': 18, '192.168.1.250': 19, '192.168.1.30': 20, '192.168.1.31': 21, '192.168.1.32': 22, '192.168.1.33': 23, '192.168.1.34': 24, '192.168.1.36': 25, '192.168.1.37': 26, '192.168.1.38': 27, '192.168.1.39': 28, '192.168.1.46': 29, '192.168.1.49': 30, '192.168.1.6': 31, '192.168.1.79': 32, '199.115.158.193': 33, '203.14.129.10': 34, '220.158.215.20': 35, '2405:6e00:10ce:2c00:20c:29ff:fed2:b002': 36, '2405:6e00:10ce:2c00:20c:29ff:feee:e07a': 37, '2405:6e00:10ce:2c00:8520:4eba:6770:702a': 38, '2405:6e00:10ce:2c00:9064:17:85d2:57d4': 39, '2405:6e00:10ce:2c00:c058:dbaa:f212:a2d8': 40, 'fe80::1087:c341:7263:ead3': 41

In [7]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


(168834, 13)
(168834,)
(42209, 13)
(42209,)


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train[:50000], y_train[:50000])
y_pred = rf.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"Random Forest Accuracy: {accuracy:.4f}")

Random Forest Accuracy: 0.9994


In [14]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=SEED, class_weight='balanced')
dt.fit(X_train[:50000], y_train[:50000])
y_pred = dt.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"Decision Tree Accuracy: {accuracy:.4f}")

Decision Tree Accuracy: 0.9983


In [15]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
lr.fit(X_train[:50000], y_train[:50000])
y_pred = lr.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"Logistic Regression Accuracy: {accuracy:.4f}")

Logistic Regression Accuracy: 0.9155


In [16]:
from sklearn.svm import SVC
svm = SVC(kernel='rbf', class_weight='balanced', random_state=SEED)
svm.fit(X_train[:50000], y_train[:50000])
y_pred = svm.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"SVM Accuracy: {accuracy:.4f}")

SVM Accuracy: 0.9944


In [17]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train[:50000], y_train[:50000])
y_pred = knn.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"KNN Accuracy: {accuracy:.4f}")

KNN Accuracy: 0.9986


In [18]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train[:50000], y_train[:50000])
y_pred = gnb.predict(X_test[:10000])
accuracy = accuracy_score(y_test[:10000], y_pred)
print(f"GaussianNB Accuracy: {accuracy:.4f}")


GaussianNB Accuracy: 0.7944


In [22]:
# Neural Network models
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Scikit-learn MLP (Multi-Layer Perceptron)
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=SEED)
mlp.fit(X_train[:3000], y_train[:3000])
y_pred = mlp.predict(X_test[:1000])
accuracy = accuracy_score(y_test[:1000], y_pred)
print(f"MLP Neural Network Accuracy: {accuracy:.4f}")

# 2. TensorFlow/Keras Deep Neural Network
def create_keras_model(input_dim, num_classes):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Train Keras model
num_classes = len(np.unique(y_train))
keras_model = create_keras_model(X_train.shape[1], num_classes)

# Train with validation split
history = keras_model.fit(X_train[:3000], y_train[:3000],
                         epochs=50,
                         batch_size=256,
                         validation_split=0.2,
                         verbose=0)

# Evaluate
y_pred_keras = keras_model.predict(X_test[:1000])
y_pred_keras = np.argmax(y_pred_keras, axis=1)
accuracy = accuracy_score(y_test[:1000], y_pred_keras)
print(f"Keras Deep Neural Network Accuracy: {accuracy:.4f}")

# 3. PyTorch Neural Network
class PyTorchNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PyTorchNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Train PyTorch model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pytorch_model = PyTorchNN(X_train.shape[1], num_classes).to(device)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train[:3000]).to(device)
y_train_tensor = torch.LongTensor(y_train[:3000]).to(device)
X_test_tensor = torch.FloatTensor(X_test[:1000]).to(device)

# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pytorch_model.parameters(), lr=0.001)

pytorch_model.train()
for epoch in range(100):
    optimizer.zero_grad()
    outputs = pytorch_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

# Evaluate PyTorch model
pytorch_model.eval()
with torch.no_grad():
    outputs = pytorch_model(X_test_tensor)
    _, y_pred_pytorch = torch.max(outputs, 1)
    y_pred_pytorch = y_pred_pytorch.cpu().numpy()

accuracy = accuracy_score(y_test[:1000], y_pred_pytorch)
print(f"PyTorch Neural Network Accuracy: {accuracy:.4f}")

MLP Neural Network Accuracy: 0.9980


/home/ritvik/Code/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1758644544.813131    7891 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0000 00:00:1758644544.958065    7891 cuda_dnn.cc:522] Loaded runtime CuDNN library: 9.1.0 but source was compiled with: 9.3.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your

FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/ritvik/Code/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/ritvik/Code/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/ritvik/Code/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/ritvik/Code/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/ritvik/Code/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 545, in dispatch_queue

  File "

In [21]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.19.0
Built with CUDA: True
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
